# NMF on ModulAir 00692

In [1]:
from sklearn.decomposition import NMF
import numpy as np
import pandas as pd

## Cleaning Data

In [2]:
#importing data from Modulair MOD-00605
data = pd.read_csv('MOD-00605.csv').set_index('timestamp')
data.head()

,id,timestamp_local,sn,rh,temp,bin0,bin1,bin2,bin3,bin4,...,no2,o3,pm1_model_id,pm25_model_id,pm10_model_id,co_model_id,no_model_id,no2_model_id,o3_model_id,ws_scalar
timestamp,,,,,,,,,,,,,,,,,,,,,
2025-12-31T05:21:09Z,577082339,2025-12-31T00:21:09Z,MOD-00605,44.1,-2.7,1.034,0.117,0.016,0.006,0.011,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,10.87
2025-12-31T05:20:20Z,577082337,2025-12-31T00:20:20Z,MOD-00605,44.7,-2.8,1.267,0.144,0.043,0.000,0.013,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,8.18
2025-12-31T05:19:20Z,577082335,2025-12-31T00:19:20Z,MOD-00605,44.6,-2.8,0.841,0.138,0.013,0.013,0.000,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,10.31
2025-12-31T05:18:20Z,577082336,2025-12-31T00:18:20Z,MOD-00605,44.4,-2.9,0.610,0.108,0.007,0.000,0.000,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,8.64
2025-12-30T03:39:14Z,576340801,2025-12-29T22:39:14Z,MOD-00605,44.3,0.5,1.078,0.210,0.030,0.003,0.015,...,NaN,NaN,16584,16585,16586,NaN,NaN,NaN,NaN,15.49


In [3]:
#only columns of interest
COLS_TO_INCLUDE = ['timestamp_local','co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
data = data[COLS_TO_INCLUDE]

data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
timestamp,,,,,,,,,,,
2025-12-31T05:21:09Z,2025-12-31T00:21:09Z,NaN,NaN,NaN,NaN,1.034,0.117,0.016,0.006,0.011,0.000
2025-12-31T05:20:20Z,2025-12-31T00:20:20Z,NaN,NaN,NaN,NaN,1.267,0.144,0.043,0.000,0.013,0.007
2025-12-31T05:19:20Z,2025-12-31T00:19:20Z,NaN,NaN,NaN,NaN,0.841,0.138,0.013,0.013,0.000,0.000
2025-12-31T05:18:20Z,2025-12-31T00:18:20Z,NaN,NaN,NaN,NaN,0.610,0.108,0.007,0.000,0.000,0.007
2025-12-30T03:39:14Z,2025-12-29T22:39:14Z,NaN,NaN,NaN,NaN,1.078,0.210,0.030,0.003,0.015,0.010


In [4]:
#removing the UTC time
data = data.reset_index(drop = True)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31T00:21:09Z,NaN,NaN,NaN,NaN,1.034,0.117,0.016,0.006,0.011,0.000
1,2025-12-31T00:20:20Z,NaN,NaN,NaN,NaN,1.267,0.144,0.043,0.000,0.013,0.007
2,2025-12-31T00:19:20Z,NaN,NaN,NaN,NaN,0.841,0.138,0.013,0.013,0.000,0.000
3,2025-12-31T00:18:20Z,NaN,NaN,NaN,NaN,0.610,0.108,0.007,0.000,0.000,0.007
4,2025-12-29T22:39:14Z,NaN,NaN,NaN,NaN,1.078,0.210,0.030,0.003,0.015,0.010


In [5]:
#converting to datetime
data['timestamp_local'] = pd.to_datetime(data['timestamp_local'],
                                       format='%Y-%m-%dT%H:%M:%SZ',
                                       exact=False)
data.head()

,timestamp_local,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-31 00:21:09,NaN,NaN,NaN,NaN,1.034,0.117,0.016,0.006,0.011,0.000
1,2025-12-31 00:20:20,NaN,NaN,NaN,NaN,1.267,0.144,0.043,0.000,0.013,0.007
2,2025-12-31 00:19:20,NaN,NaN,NaN,NaN,0.841,0.138,0.013,0.013,0.000,0.000
3,2025-12-31 00:18:20,NaN,NaN,NaN,NaN,0.610,0.108,0.007,0.000,0.000,0.007
4,2025-12-29 22:39:14,NaN,NaN,NaN,NaN,1.078,0.210,0.030,0.003,0.015,0.010


In [6]:
#taking hourly average of df. round to floor of the hour
data = data.groupby(data['timestamp_local'].dt.floor('h')).agg(co = ('co','mean'),
                                                         no2 = ('no2','mean'),
                                                         o3 = ('o3','mean'),
                                                         no = ('no','mean'),
                                                         bin0 = ('bin0','mean'),
                                                         bin1 = ('bin1','mean'),
                                                         bin2 = ('bin2','mean'),
                                                         bin3 = ('bin3','mean'),
                                                         bin4 = ('bin4','mean'),
                                                         bin5 = ('bin5','mean')).reset_index()

data = data.round(decimals = 2)
data = data.dropna()
data

,timestamp_local,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
0,2025-12-09 04:00:00,1385.97,34.01,60.05,2.74,2.30,0.33,0.10,0.02,0.04,0.03
1,2025-12-09 05:00:00,1099.04,34.40,57.60,2.76,2.34,0.30,0.07,0.02,0.02,0.01
2,2025-12-09 06:00:00,1048.72,33.96,59.04,2.85,2.39,0.31,0.09,0.02,0.03,0.02
3,2025-12-09 07:00:00,1394.84,33.70,60.81,2.93,1.65,0.26,0.07,0.01,0.02,0.01
4,2025-12-09 08:00:00,1352.87,34.91,60.37,2.45,2.70,0.37,0.10,0.02,0.03,0.02
...,...,...,...,...,...,...,...,...,...,...,...
415,2025-12-27 16:00:00,716.04,32.70,58.77,1.90,2.59,0.44,0.13,0.03,0.03,0.02
416,2025-12-27 17:00:00,737.84,33.36,58.69,1.79,2.78,0.50,0.15,0.03,0.03,0.01
417,2025-12-27 18:00:00,735.48,32.40,59.04,1.82,2.59,0.45,0.13,0.02,0.02,0.01
418,2025-12-27 19:00:00,715.52,32.42,59.37,1.64,3.00,0.48,0.13,0.02,0.03,0.01


In [7]:
#setting local time as index
data = data.set_index('timestamp_local')
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-12-09 04:00:00,1385.97,34.01,60.05,2.74,2.30,0.33,0.10,0.02,0.04,0.03
2025-12-09 05:00:00,1099.04,34.40,57.60,2.76,2.34,0.30,0.07,0.02,0.02,0.01
2025-12-09 06:00:00,1048.72,33.96,59.04,2.85,2.39,0.31,0.09,0.02,0.03,0.02
2025-12-09 07:00:00,1394.84,33.70,60.81,2.93,1.65,0.26,0.07,0.01,0.02,0.01
2025-12-09 08:00:00,1352.87,34.91,60.37,2.45,2.70,0.37,0.10,0.02,0.03,0.02


In [8]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()

In [9]:
#scaling
def maximum_absolute_scaling(df):
    # copy the dataframe
    df_scaled = df.copy()
    # apply maximum absolute scaling
    for column in df_scaled.columns:
        df_scaled[column] = df_scaled[column]  / df_scaled[column].abs().max()
    return df_scaled

# call the maximum_absolute_scaling function
data = maximum_absolute_scaling(data)

In [10]:
#subsetting for only positive and non NA values
data = data.where(data>0)
data = data.dropna()
data.head()

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-12-09 04:00:00,0.748299,0.840791,0.926126,0.073597,0.077285,0.030698,0.050,0.046512,0.057971,0.047619
2025-12-09 05:00:00,0.593383,0.850433,0.888341,0.074134,0.078629,0.027907,0.035,0.046512,0.028986,0.015873
2025-12-09 06:00:00,0.566215,0.839555,0.910549,0.076551,0.080309,0.028837,0.045,0.046512,0.043478,0.031746
2025-12-09 07:00:00,0.753088,0.833127,0.937847,0.078700,0.055444,0.024186,0.035,0.023256,0.028986,0.015873
2025-12-09 08:00:00,0.730428,0.863041,0.931061,0.065807,0.090726,0.034419,0.050,0.046512,0.043478,0.031746


In [11]:
data.to_csv(r'MOD-000605_timeseries_hourly_scaled.csv')

## Implementing NMF

In [12]:
#setting up the NMF
nmf = NMF(n_components = 4, max_iter = 8000)

In [13]:
W = nmf.fit_transform(data)
H = nmf.components_
V = pd.DataFrame(np.dot(W,H), columns=data.columns)
V.index = data.index
V

,co,no2,o3,no,bin0,bin1,bin2,bin3,bin4,bin5
timestamp_local,,,,,,,,,,
2025-12-09 04:00:00,0.579335,0.918933,0.931489,0.098417,0.177416,0.003574,0.031779,0.068378,0.058755,0.045988
2025-12-09 05:00:00,0.522582,0.868025,0.904221,0.080781,0.130355,0.002457,0.022106,0.043940,0.035740,0.025483
2025-12-09 06:00:00,0.512031,0.866054,0.911497,0.075389,0.114264,0.003182,0.028401,0.053978,0.045501,0.035746
2025-12-09 07:00:00,0.571066,0.920164,0.941559,0.094774,0.165974,0.001952,0.017753,0.039420,0.030868,0.018990
2025-12-09 08:00:00,0.582357,0.926127,0.940231,0.098727,0.177337,0.003013,0.026933,0.058781,0.049451,0.036795
...,...,...,...,...,...,...,...,...,...,...
2025-12-27 16:00:00,0.411142,0.801863,0.901682,0.038700,0.073642,0.053629,0.074221,0.060207,0.042452,0.035275
2025-12-27 17:00:00,0.418617,0.810970,0.908659,0.040914,0.086100,0.058914,0.076510,0.057494,0.038645,0.031044
2025-12-27 18:00:00,0.411270,0.802882,0.902978,0.039154,0.076965,0.054353,0.066834,0.043519,0.026000,0.018962


In [14]:
W_df = pd.DataFrame(W, columns =[f'Feature {i+1}' for i in range(0,4)]) #array-like of shape (n_samples, n_components)
W_df

,Feature 1,Feature 2,Feature 3,Feature 4
0,0.171810,0.000000,0.035077,0.304524
1,0.171040,0.000000,0.017594,0.223747
2,0.174154,0.000000,0.028625,0.196127
3,0.175033,0.000000,0.009152,0.284886
4,0.173603,0.000000,0.026030,0.304388
...,...,...,...,...
372,0.180990,0.029494,0.034038,0.020719
373,0.181671,0.032773,0.029550,0.030350
374,0.181086,0.030524,0.017908,0.022733
375,0.181471,0.034509,0.020605,0.017826


In [15]:
COLS_TO_INCLUDE.pop(0)
COLS_TO_INCLUDE

['co', 'no', 'no2', 'o3', 'bin0', 'bin1', 'bin2', 'bin3', 'bin4', 'bin5']

In [16]:
H_df = pd.DataFrame(H, index = [f'Feature {i+1}' for i in range(0,4)], columns = COLS_TO_INCLUDE) #array-like of shape (n_components, n_features)
H_df

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Feature 1,2.179237,4.313515,4.906488,0.189034,0.000000,0.007831,0.072662,0.054802,0.022557,0.000000
Feature 2,0.000000,0.252572,0.257148,0.000000,2.087602,1.696944,1.435773,0.420154,0.066788,0.000000
Feature 3,0.087835,0.050966,0.001636,0.000000,0.000000,0.063533,0.550086,1.070550,1.032028,1.015615
Feature 4,0.662803,0.578087,0.290450,0.216532,0.582600,0.000000,0.000000,0.070308,0.061339,0.034031


In [17]:
#converting the results to a dataframe
results = pd.DataFrame(W,index=data.index) #H is time series data of the factors, W is the comp (coeff matrix)
results.columns = ["Factor {}".format(i+1) for i in range(H.T.shape[1])]
results

,Factor 1,Factor 2,Factor 3,Factor 4
timestamp_local,,,,
2025-12-09 04:00:00,0.171810,0.000000,0.035077,0.304524
2025-12-09 05:00:00,0.171040,0.000000,0.017594,0.223747
2025-12-09 06:00:00,0.174154,0.000000,0.028625,0.196127
2025-12-09 07:00:00,0.175033,0.000000,0.009152,0.284886
2025-12-09 08:00:00,0.173603,0.000000,0.026030,0.304388
...,...,...,...,...
2025-12-27 16:00:00,0.180990,0.029494,0.034038,0.020719
2025-12-27 17:00:00,0.181671,0.032773,0.029550,0.030350
2025-12-27 18:00:00,0.181086,0.030524,0.017908,0.022733


In [18]:
COLS_TO_INCLUDE = ['co','no','no2','o3','bin0','bin1','bin2','bin3','bin4','bin5']
comp = pd.DataFrame(H, index = results.columns, columns = COLS_TO_INCLUDE)
comp

,co,no,no2,o3,bin0,bin1,bin2,bin3,bin4,bin5
Factor 1,2.179237,4.313515,4.906488,0.189034,0.000000,0.007831,0.072662,0.054802,0.022557,0.000000
Factor 2,0.000000,0.252572,0.257148,0.000000,2.087602,1.696944,1.435773,0.420154,0.066788,0.000000
Factor 3,0.087835,0.050966,0.001636,0.000000,0.000000,0.063533,0.550086,1.070550,1.032028,1.015615
Factor 4,0.662803,0.578087,0.290450,0.216532,0.582600,0.000000,0.000000,0.070308,0.061339,0.034031


In [19]:
res = []

for col in comp.columns:
    #for each factor, compute its contribution to the species in V
    by_factor = pd.Series(dtype=float)
    for i, factor in enumerate(results.columns):
        contrib = H[i, data.columns.get_loc(col)] * W[:, i].sum()
        by_factor.at[factor] = contrib

    #normalizing by the total amount of that species in the original data
    by_factor /= data[col].sum()

    #adding as a row to the resulting dataframe
    res.append(pd.DataFrame(by_factor).T.rename(index={0: col}))

res = pd.concat(res)
res.columns = results.columns
res['Residual'] = 1 - res.sum(axis=1)

res

,Factor 1,Factor 2,Factor 3,Factor 4,Residual
co,0.761227,0.000000,0.008634,0.230667,-0.000527
no,0.469365,0.000000,0.000000,0.535653,-0.005017
no2,0.867488,0.015088,0.002884,0.115829,-0.001289
o3,0.929437,0.014469,0.000087,0.054817,0.001190
bin0,0.000000,0.516963,0.000000,0.483909,-0.000872
bin1,0.014887,0.958180,0.033984,0.000000,-0.007051
bin2,0.111852,0.656503,0.238275,0.000000,-0.006630
bin3,0.099340,0.226228,0.546063,0.126978,0.001391
bin4,0.057141,0.050255,0.735652,0.154812,0.002140
bin5,0.000000,0.000000,0.901075,0.106902,-0.007977


In [20]:
results.to_csv(r'4_factor_results.csv')
comp.to_csv(r'4_factor_comp.csv')
res.to_csv(r'4_factor_resid.csv')